# C-MAPSS RUL: thin orchestration notebook

**All logic lives in `src/` — this notebook only installs, imports, overrides config, calls functions, and plots.**
See `RESEARCH_PLAN.md` for scope and protocol, and `CHANGES.md` for every deviation.

## How to run (in order)
1. **Setup** — installs + mount Drive + `sys.path` + GPU check.
2. **Config** — point paths at your Drive; override any design decision here.
3. **Stage A (GPU, once per embedding config)** — Chronos-2 `embed()` → versioned cache on Drive (pooled embeddings **+ per-window loc/scale + fixed raw windows**). Variable-length TSFM contexts are fed to `embed()` natively (short test histories are left-pad-**masked**, not fabricated). *Idempotent*: re-running skips if the cache key exists. Degrades to a T4 via `embed_batch_size`/`embed_dtype`.
4. **Stage A2 (ablation)** — full-data, MSE, 3 seeds over `tsfm_context_length × head_features` (+ the raw-fusion arm and pooling variants at the best cell). Picks the winning **(context, features, pooling)** cell; builds one Stage A cache per context/pooling as needed.
5. **Stage B (cheap, rerunnable)** — data-fraction × loss × seed sweep + baselines at the **winning** config, on the **cached** arrays. Writes `results_v2.csv` with **both** test-label protocols (clipped + unclipped). Checkpoints every cell; never re-embeds.
6. **Stage C** — load result CSVs and plot the learning curves + the data-scaling curve.

The economics: **embeddings once per config (Stage A), everything downstream free (Stages A2/B).**

## 1. Setup

In [ ]:
# Pinned installs. Colab already ships torch/numpy/pandas/scikit-learn/matplotlib.
# chronos-forecasting 2.x provides the official Chronos2Pipeline.embed().
%pip install -q "chronos-forecasting>=2.0.0" "coral-pytorch==1.4.0" "lightgbm>=4.0" "sktime>=1.0" "numba>=0.59" "safetensors>=0.4"

import sys, os, torch
from google.colab import drive

# Mount Drive (holds CMAPSSData/, the embedding cache, and results).
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Add the repo (kept on Drive) to sys.path so `import src.*` works.
REPO_DIR = '/content/drive/MyDrive/Predictive Maintenance LSTM/Predictive-Maintenance-LSTM'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Config

The single source of truth is `src/config.py`. Override only paths (and any design decision you want to ablate) here.

**Multi-dataset runs (CHANGES.md §21-22):** set `DATASET` below and rerun the notebook top-to-bottom — every cache, results row, and figure is keyed by dataset, so runs coexist in the same `results_v2.csv` / `horizon.csv` / `figures/`.
- `FD001`/`FD003` (single condition): unchanged behavior.
- `FD002`/`FD004` (6 operating conditions): condition-wise normalization turns ON automatically (per-condition z-norm, stats fit on the train split).
- `XJTU-SY` bearings: point `data_dir` at the XJTU-SY root (3 condition folders) and set `sensor_columns=XJTU_FEATURE_COLUMNS`; "cycles" are 1-minute snapshots, so pick `max_rul` deliberately (it is in minutes).

NOTE: adding `condition_norm` to the cache key invalidates all pre-§21 Stage A caches — the first run after this change re-embeds (one Stage A pass per dataset).

In [ ]:
from src.config import Config

DRIVE = '/content/drive/MyDrive/Predictive Maintenance LSTM'
DATASET = 'FD001'          # FD001 | FD002 | FD003 | FD004 | XJTU-SY

config = Config(
    dataset=DATASET,
    data_dir=f'{DRIVE}/CMAPSSData',   # for XJTU-SY: f'{DRIVE}/XJTU-SY' (3 condition folders)
    cache_dir=f'{DRIVE}/cache',
    results_dir=f'{DRIVE}/results',
    # --- multi-dataset knobs ---
    # condition_norm=None,              # None => auto (ON for FD002/FD004/XJTU-SY)
    # sensor_columns=...,               # XJTU-SY: from src.xjtu import XJTU_FEATURE_COLUMNS
    # --- ablation / GPU knobs (all optional; Stage A2 sweeps these for you) ---
    # tsfm_context_length=120,          # TSFM history, independent of window_size (None => window_size)
    # head_features='emb+locscale',     # emb | emb+locscale | emb+locscale+raw
    # pooling='mean',                   # forecast_token | last_content | mean | flatten
    # embed_batch_size=64,              # lower for a T4
    # embed_dtype='float16',            # embed() compute dtype: bfloat16 | float16 | float32
    # embedding_storage_dtype='float32',# on-disk emb dtype (default float16; halves Drive I/O)
    # losses=['mse', 'corn', 'quantile'],
)
config


## 3. Stage A — Embedding pass (GPU, run once per embedding config)

Loads FD001, builds **fixed** baseline windows (for the baselines + the raw-fusion last-cycle sensors) **and** variable-length TSFM contexts aligned 1:1 to them, runs Chronos-2 `embed()` batched on GPU, and caches: pooled embeddings (**float16**), per-window **loc/scale** (float32), fixed raw windows (float32), labels, unit IDs. Keyed by `config.embedding_cache_key()` (window size, pooling, model, **tsfm_context_length**, **schema version**). **Idempotent** — if the key exists it loads instead of recomputing. Throughput (windows/s) is printed and stored in the sidecar.

> Running the *ablation* (Stage A2) next builds these caches for you per (context, pooling). Run this cell directly only for a single explicit config.

In [ ]:
from src.embeddings import build_embedding_cache

cache_path = build_embedding_cache(config)   # skips instantly if the key already exists
print('embedding cache:', cache_path)
print('sidecar        :', cache_path.with_suffix('.json').read_text())

## 3b. Stage A2 — Ablation (pick the winning config)

Full-data, MSE, 3 seeds over `tsfm_context_length ∈ {30,60,120,256} × head_features ∈ {emb, emb+locscale}`, then the `emb+locscale+raw` arm and the `{mean, last_content}` pooling variants at the best cell. Each (context, pooling) triggers one Stage A cache build (idempotent). Restartable — rows are checkpointed to `ablation.csv`. `select_best_ablation_cell` picks the winner by **clipped RMSE** (the literature-comparable metric). Record the winner + justification in `CHANGES.md` §9.

In [ ]:
from src.sweep import run_ablation, select_best_ablation_cell
import torch, pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ablation_csv = run_ablation(config, device=device)   # builds caches + trains heads over the grid

best = select_best_ablation_cell(ablation_csv)        # winner by clipped RMSE
print('WINNING CELL:', best)

# Full ablation table (mean ± std clipped RMSE over seeds), sorted best-first.
abl = pd.read_csv(ablation_csv)
summary = (abl.groupby(['tsfm_context_length', 'head_features', 'pooling'])['rmse_clipped']
              .agg(['mean', 'std', 'count']).sort_values('mean'))
summary

## 4. Stage B — Sweep at the winning config (cheap, rerunnable)

Data-fraction × loss × seed MLP heads at the **winning** `(context, head_features, pooling)` from Stage A2, plus baselines on the cached raw windows. Consumes the cache only — **no re-embedding**. Writes `results_v2.csv` with **both** test-label protocols (`rmse_clipped`/`…_unclipped`); any pre-existing `results.csv` is archived to `results_v1.csv` first. Appends after every cell and skips completed cells on restart.

In [ ]:
from src.sweep import run_sweep, run_baseline_window_comparison
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'   # heads train on-GPU (tensor slicing)

# Adopt the ablation winner. (Also builds its Stage A cache if not already present.)
sweep_config = config.replace(
    tsfm_context_length=int(best['tsfm_context_length']),
    head_features=best['head_features'],
    pooling=best['pooling'],
)
from src.embeddings import build_embedding_cache
build_embedding_cache(sweep_config)                       # idempotent; no-op if Stage A2 built it

# Optional (Task 1.5): compare GBM/LSTM at window 30 vs 120 at full data. If a longer
# window wins, adopt it per baseline, e.g. sweep_config = sweep_config.replace(
#     baseline_windows={'lstm': 120}).  Then baselines are re-windowed from the raw series.
# run_baseline_window_comparison(config, device=device)

results_csv = run_sweep(sweep_config, device=device)
print('results CSV:', results_csv)

### 4b. Fairness arms — hand the baselines the age signal (CHANGES.md §19)

The winning TSFM config sees up to 256 cycles while baselines see 30, and its
variable-length context implicitly carries engine age (CHANGES.md §12 caveat 2).
These arms bound how much of the gap that explains:
- `run_baseline_window_comparison`: GBM/LSTM at windows {30, 60, 120} — if a longer
  window wins at full data, adopt it via `sweep_config.replace(baseline_windows=...)`
  and rerun the sweep cell.
- `run_fairness_baselines`: the `cycle_reg` age floor (plan §4) + `gbm_age` (GBM with
  `time_cycles` as an extra channel). Rows land in `results_v2.csv`, so the
  data-scaling figure picks them up automatically.

In [ ]:
from src.sweep import run_baseline_window_comparison, run_fairness_baselines
import pandas as pd

bw_csv = run_baseline_window_comparison(sweep_config, windows=[30, 60, 120], device=device)
print(pd.read_csv(bw_csv).groupby(['model', 'baseline_window'])['rmse_clipped']
        .agg(['mean', 'std']).round(2))
# If a longer window clearly wins for a baseline, adopt it and rerun the sweep:
# sweep_config = sweep_config.replace(baseline_windows={'lstm': 120}); run_sweep(...)

run_fairness_baselines(sweep_config)   # cycle_reg + gbm_age -> results_v2.csv (CPU)


## 5. Stage C — Results

### Data-scaling curve (the headline figure): test error vs. training units, one line per model, error bands over seeds. Plotted for the **clipped** protocol (literature-comparable) and the **unclipped** protocol.

In [ ]:
# Stage C figures (src/plots.py): saved to <results_dir>/figures/ as 300-dpi PNG + PDF.
# predict_mean is a gray reference line (no longer squashes the y-axis); NASA panels are log-y.
from pathlib import Path
from src.plots import plot_data_scaling, plot_ablation

figures_dir = Path(config.results_dir) / 'figures'
saved = plot_data_scaling(results_csv, figures_dir)
saved += plot_ablation(ablation_csv, figures_dir)
print('saved:', *[str(s) for s in saved], sep='\n  ')


### Learning curves: validation RMSE vs. epoch, per sweep cell.

In [ ]:
# Learning curves: one panel per loss arm, shaded by training-unit count (darker = more units).
from src.plots import plot_learning_curves

saved = plot_learning_curves(Path(config.results_dir) / 'runs' / 'learning_curves', figures_dir)
print('saved:', *[str(s) for s in saved], sep='\n  ')


## 6. Horizon-stratified evaluation — error vs. distance-to-failure

The main protocol scores one prediction per test unit (its last cycle), so far-from-failure
behavior — the predictions that buy planning time — is invisible. This embeds EVERY test
cycle (same context construction as training rows, separate cache; the main cache stays
valid) and stratifies error by the unclipped true RUL.

**Reading it:** bins below `max_rul` measure real accuracy at that horizon; the `≥125` bin
only measures saturation quality (training labels are clipped, so no model can express
horizons beyond the cap — see `src/horizon.py` docstring / CHANGES.md §16).

In [ ]:
from src.horizon import build_horizon_cache, run_horizon_eval

build_horizon_cache(sweep_config)   # GPU: embeds every test cycle once (idempotent)
horizon_csv = run_horizon_eval(sweep_config, n_units_list=[10, 100], device=device)
print('horizon CSV:', horizon_csv)


In [ ]:
from src.plots import plot_horizon, plot_horizon_trajectories

saved = plot_horizon(horizon_csv, figures_dir)
saved += plot_horizon_trajectories(
    Path(sweep_config.results_dir) / 'horizon_predictions.csv', figures_dir,
    max_rul=sweep_config.max_rul)
print('saved:', *[str(s) for s in saved], sep='\n  ')


### 6a. CORN vs MSE per horizon bin — paired significance (CHANGES.md §18)

The horizon eval now defaults to all 5 seeds. Pairing by seed is valid (both loss arms
share each seed's sampled units and split). With 5 seeds the test is low-powered —
read p-values as descriptive, alongside the per-bin means.

In [ ]:
from src.evaluate import paired_seed_ttest
import pandas as pd

sig = pd.DataFrame(paired_seed_ttest(horizon_csv, loss_a='corn', loss_b='mse',
                                     metric='mae_clipped'))
sig[['mean_corn', 'mean_mse', 'mean_delta', 't', 'p']] = \
    sig[['mean_corn', 'mean_mse', 'mean_delta', 't', 'p']].round(3)
sig  # mean_delta < 0 => CORN better in that bin


## 6b. Raised label cap — genuine long-horizon skill (max_rul = 200)

The 125-cap runs above stay untouched (literature-comparable). This arm re-keys the
caches (labels are cached with the windows, so it is a fresh Stage A pass) and re-runs
the horizon eval at `max_rul=200`; bins extend automatically to {125–150, 150–175,
175–200, ≥200}. Both cap arms share `horizon.csv` / `horizon_predictions.csv`
(`max_rul` is part of the row key). Bins **below 125 are directly comparable across
arms** — if the 200-cap model gets worse there, the extra horizon costs near-failure
accuracy; the 125–200 bins measure whether degradation is detectable that early at all
(CHANGES.md §18).

NOTE: if a pre-existing `horizon_predictions.csv` predates the `max_rul` column, the
run stops with a schema error — move the old file into e.g. `results/archive/` first.

In [ ]:
mr200_config = sweep_config.replace(max_rul=200)

build_embedding_cache(mr200_config)      # fresh Stage A pass (new cache key)
build_horizon_cache(mr200_config)        # GPU: every test cycle at the new key
horizon200_csv = run_horizon_eval(mr200_config, n_units_list=[10, 100], device=device)
print('horizon CSV (both cap arms):', horizon200_csv)


In [ ]:
saved = plot_horizon(horizon200_csv, figures_dir)   # one figure per (cap, n_units)
saved += plot_horizon_trajectories(
    Path(mr200_config.results_dir) / 'horizon_predictions.csv', figures_dir,
    max_rul=200)
print('saved:', *[str(s) for s in saved], sep='\n  ')


## 7. Cold-start transfer — deploy with zero target failures

Can a head trained on an analogous fleet (source) produce useful RUL on a fleet with **no
recorded failures** (target), and how fast do a few target failures close the gap? Three
arms per shot count: `zero_shot` (source-only), `target_only`, `source+target`
(CHANGES.md §17). FD001↔FD003 is the a-priori-valid pair (both single-condition, same
non-constant sensors); FD002/FD004 pairs are now supported through condition-wise
normalization, fit per dataset on its own train split (CHANGES.md §21). Builds the target's Stage A cache on first run (GPU).

In [ ]:
from src.transfer import run_transfer_eval
from src.plots import plot_transfer

transfer_csv = run_transfer_eval(sweep_config, source_dataset='FD001',
                                 target_dataset='FD003',
                                 shots=[2, 5, 10, 25], device=device)
saved = plot_transfer(transfer_csv, figures_dir)
print('saved:', *[str(s) for s in saved], sep='\n  ')
